# Cleaning Downloaded Data from avian-flu

Author: Alexander Maksiaev

Purpose: Clean downloaded data from avian-flu, rename sequences according to convention, de-duplicate from GISAID

In [1]:
# Housekeeping

import os
import pandas as pd
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 


# Dates
start_date = "04-14-2025"
end_date = "06-13-2025"
date_range = start_date + "--" + end_date

# Make sure you have the correct paths

home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/"
downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
# home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/"
# downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
references = home + "references/"
originals = downloads + "Andersen/"
saved = originals + "saved/"
temp_files = originals + "temp/"
complete_files = downloads + "complete/"

os.chdir(downloads)

## Read Metadata 

In [2]:
# Read metadata

# os.chdir(saved)
# metadata_normalized = pd.read_csv("metadata_normalized.tsv", delimiter="\t") # Collection dates

metadata_folder = originals + "avian-influenza/metadata/"
os.chdir(metadata_folder)

metadata = pd.read_csv("SraRunTable_automated.csv")

print(len(metadata)) 

# metadata = metadata.merge(metadata_normalized, how="outer")
print(metadata.columns)

# Find the name of the state sample was collected in
metadata["name_state"] = metadata["geo_loc_name"].apply(lambda x: x.split("/")[1] if len(x.split("/")) > 1 else x.split("/")[0])

# Convert the dates to date format so we can compare
metadata["ReleaseDate"] = metadata["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))

metadata = metadata[metadata["ReleaseDate"] >= dateutil.parser.parse(start_date).strftime("%Y-%m-%d")] # Find only >= last date using Release Date from metadata 
metadata = metadata[metadata["ReleaseDate"] <= dateutil.parser.parse(end_date).strftime("%Y-%m-%d")] # Find only <= update date using Release Date from metadata

# # Get rid of certain runs
# os.chdir(saved)
# runs_to_remove = pd.read_csv("andersen-lab-seqs-to-filter.csv")
# print(runs_to_remove)
# for run in runs_to_remove["Run"].values:
#     metadata = metadata[metadata["Run"] != run]

print(len(metadata)) 
display(metadata)
print(metadata["Library Name"])

9899
Index(['Run', 'Assay Type', 'AvgSpotLen', 'Bases', 'BioProject', 'BioSample',
       'BioSampleModel', 'Bytes', 'Center Name', 'Collection_Date', 'Consent',
       'DATASTORE filetype', 'DATASTORE provider', 'DATASTORE region',
       'Experiment', 'geo_loc_name_country', 'geo_loc_name_country_continent',
       'geo_loc_name', 'Host', 'Instrument', 'isolate', 'Library Name',
       'LibraryLayout', 'LibrarySelection', 'LibrarySource', 'Organism',
       'Platform', 'ReleaseDate', 'create_date', 'version', 'Sample Name',
       'SRA Study', 'serotype', 'isolation_source', 'BioSample Accession',
       'is_retracted', 'retraction_detection_date_utc'],
      dtype='object')
1371


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,create_date,version,Sample Name,SRA Study,serotype,isolation_source,BioSample Accession,is_retracted,retraction_detection_date_utc,name_state
8388,SRR33125012,WGS,148.14,103837402,PRJNA1207547,SAMN47941494,Viral,38567600,USDA-NVSL,2025,...,2025-04-14 15:01:46,1,25-004426-001,SRP557452,NaN,CLOACAL/OROPHARYNGEAL SWAB POOL,SRS24712538,False,NaN,USA
8389,SRR33125013,WGS,148.47,151899011,PRJNA1207547,SAMN47941493,Viral,55900173,USDA-NVSL,2025,...,2025-04-14 14:59:12,1,25-004499-002,SRP557452,NaN,CLOACAL/OROPHARYNGEAL SWAB POOL,SRS24712537,False,NaN,USA
8390,SRR33125014,WGS,147.45,166656358,PRJNA1207547,SAMN47941492,Viral,60793010,USDA-NVSL,2025,...,2025-04-14 14:59:27,1,25-004497-005,SRP557452,NaN,CLOACAL/OROPHARYNGEAL SWAB POOL,SRS24712536,False,NaN,USA
8391,SRR33125015,WGS,148.33,107664413,PRJNA1207547,SAMN47941491,Viral,39666477,USDA-NVSL,2025,...,2025-04-14 15:00:01,1,25-004497-004,SRP557452,NaN,CLOACAL/OROPHARYNGEAL SWAB POOL,SRS24712534,False,NaN,USA
8392,SRR33125016,WGS,120.38,2476776,PRJNA1207547,SAMN47941490,Viral,986034,USDA-NVSL,2025,...,2025-04-14 14:59:20,1,25-004480-001,SRP557452,NaN,tracheal swab,SRS24712535,False,NaN,USA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9754,SRR33943358,WGS,148.58,145468815,PRJNA1102327,SAMN49008963,Viral,50309372,USDA-NVSL,2025,...,2025-06-11 17:51:42,1,25-013594-001,SRP503016,NaN,milk,SRS25347775,False,NaN,USA
9755,SRR33943359,WGS,148.65,141682464,PRJNA1102327,SAMN49008954,Viral,52273363,USDA-NVSL,2025,...,2025-06-11 17:51:39,1,25-016317-001,SRP503016,NaN,milk,SRS25347774,False,NaN,USA
9756,SRR33943360,WGS,148.56,109583296,PRJNA1102327,SAMN49008953,Viral,40873907,USDA-NVSL,2025,...,2025-06-11 17:51:39,1,25-015080-001,SRP503016,NaN,"MILK, BULK TANK",SRS25347773,False,NaN,USA
9757,SRR33943301,WGS,148.82,146047742,PRJNA980729,SAMN49008919,Viral,51264801,USDA-NVSL,2025,...,2025-06-11 17:50:24,1,25-016837-002,SRP441379,NaN,OROPHARYNGEAL SWAB,SRS25347770,False,NaN,USA


8388            25-004426-001-original
8389            25-004499-002-original
8390            25-004497-005-original
8391            25-004497-004-original
8392            25-004480-001-original
                     ...              
9754    25-013594-001-original-repeat2
9755            25-016317-001-original
9756            25-015080-001-original
9757            25-016837-002-original
9758            25-016837-001-original
Name: Library Name, Length: 1371, dtype: object


In [3]:
# Get list of genotypes

os.chdir(home + "references/")

# genotypes_df = pd.read_excel("genotype_key.xlsx")

# genotypes = list(genotypes_df["Genotype"])

# print(genotypes)

genotypes = ["B3.13", "D1.1"]

# genotypes = ["B3.2"] #, "B3.2", "B3.6", "B3.7", "B3.5", "A3"]

### Naming convention ###
>A/[host]/[geo_loc_name]/[isolate]/[year]|[serotype: H5N1]|[collection_date]|[host_type]|[genotype]

host_type is from manual animal reference

In metadata, we have: host, geo_loc_name, isolate, year

We need: geo_loc_name, collection_date, host_type, genotype

host = Host

geo_loc_name (primary) = geo_loc_name

geo_loc_name (secondary) = genbank_mapping.tsv > genbank_name

isolate = isolate

collection date (primary) = Collection_Date

collection date (secondary) = https://www.ncbi.nlm.nih.gov/genbank/ > BioSample (input: BioSample) > Nucleotide > [first result] > collection_date

serotype = serotype

host type = [from ref] 

genotype = [from genoflu] -- use genoflu_results.tsv

## Get genotype, specific geolocation

In [4]:
# Get genotype from genoflu_results.tsv

os.chdir(metadata_folder)

genoflu_results = pd.read_csv("genoflu_results.tsv", delimiter="\t")

genoflu_results["Run"] = genoflu_results["sample"]

metadata = metadata.merge(genoflu_results, on="Run", how="inner")
print(metadata)
# metadata = metadata[~metadata["Genotype"].str.contains('Not assigned')] # Do not include non-assigned genotypes
metadata = metadata[metadata["Genotype"].isin(genotypes)]

print(len(metadata)) 
display(metadata)

              Run Assay Type  AvgSpotLen      Bases    BioProject  \
0     SRR33125012        WGS      148.14  103837402  PRJNA1207547   
1     SRR33125013        WGS      148.47  151899011  PRJNA1207547   
2     SRR33125014        WGS      147.45  166656358  PRJNA1207547   
3     SRR33125015        WGS      148.33  107664413  PRJNA1207547   
4     SRR33125016        WGS      120.38    2476776  PRJNA1207547   
...           ...        ...         ...        ...           ...   
1366  SRR33943358        WGS      148.58  145468815  PRJNA1102327   
1367  SRR33943359        WGS      148.65  141682464  PRJNA1102327   
1368  SRR33943360        WGS      148.56  109583296  PRJNA1102327   
1369  SRR33943301        WGS      148.82  146047742   PRJNA980729   
1370  SRR33943302        WGS      148.87  176312039   PRJNA980729   

         BioSample BioSampleModel     Bytes Center Name Collection_Date  ...  \
0     SAMN47941494          Viral  38567600   USDA-NVSL            2025  ...   
1     SAMN4

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,name_state,sample,date,File Name,Genotype,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List
0,SRR33125012,WGS,148.14,103837402,PRJNA1207547,SAMN47941494,Viral,38567600,USDA-NVSL,2025,...,USA,SRR33125012,2025-05-09_10-52-38,SRR33125012.fa,D1.1,"NP:am13, NS:ea3, PA:am4, PB2:am24, MP:ea3, PB1...","am13:24-030039-001:NP, ea3:22-013001-001:NS, a...","99.87%, 98.93%, 99.72%, 99.61%, 99.80%, 99.39%...","2, 9, 6, 9, 2, 11, 8, 12",Ran on FASTA - No Coverage Report
1,SRR33125013,WGS,148.47,151899011,PRJNA1207547,SAMN47941493,Viral,55900173,USDA-NVSL,2025,...,USA,SRR33125013,2025-05-09_10-52-46,SRR33125013.fa,D1.1,"NA:am4N1, NS:ea3, PB2:am24, MP:ea3, PB1:ea3, P...","am4N1:24-030039-001:NA, ea3:22-013001-001:NS, ...","98.87%, 99.17%, 99.83%, 99.90%, 99.16%, 99.58%...","13, 7, 4, 1, 19, 9, 9, 2",Ran on FASTA - No Coverage Report
2,SRR33125014,WGS,147.45,166656358,PRJNA1207547,SAMN47941492,Viral,60793010,USDA-NVSL,2025,...,USA,SRR33125014,2025-05-09_10-49-25,SRR33125014.fa,D1.1,"PB2:am24, MP:ea3, PB1:ea3, NP:am13, NA:am4N1, ...","am24:24-030039-001:PB2, ea3:22-013001-001:MP, ...","99.83%, 99.90%, 99.25%, 99.87%, 98.96%, 99.41%...","4, 1, 17, 2, 12, 10, 6, 11",Ran on FASTA - No Coverage Report
3,SRR33125015,WGS,148.33,107664413,PRJNA1207547,SAMN47941491,Viral,39666477,USDA-NVSL,2025,...,USA,SRR33125015,2025-05-09_10-48-50,SRR33125015.fa,D1.1,"PA:am4, NA:am4N1, NP:am13, PB1:ea3, NS:ea3, MP...","am4:24-030039-001:PA, am4N1:24-030039-001:NA, ...","99.58%, 99.03%, 99.87%, 98.99%, 99.28%, 99.90%...","9, 11, 2, 23, 6, 1, 3, 9",Ran on FASTA - No Coverage Report
4,SRR33125016,WGS,120.38,2476776,PRJNA1207547,SAMN47941490,Viral,986034,USDA-NVSL,2025,...,USA,SRR33125016,2025-05-09_10-52-46,SRR33125016.fa,D1.1,"NA:am4N1, PB1:ea3, NS:ea3, MP:ea3, NP:am13, PA...","am4N1:24-030039-001:NA, ea3:22-013001-001:PB1,...","99.62%, 99.28%, 99.28%, 99.69%, 99.93%, 98.34%...","4, 7, 6, 3, 1, 29, 9, 9",Ran on FASTA - No Coverage Report
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1366,SRR33943358,WGS,148.58,145468815,PRJNA1102327,SAMN49008963,Viral,50309372,USDA-NVSL,2025,...,USA,SRR33943358,2025-06-13_07-18-34,SRR33943358.fa,B3.13,"PA:ea1, NA:ea1, PB2:am2.2, NP:am8, HA:ea1, PB1...","ea1:22-003707-003:PA, ea1:22-003707-003:NA, am...","98.84%, 98.65%, 98.51%, 98.86%, 98.12%, 99.56%...","25, 19, 34, 17, 32, 10, 9, 10",Ran on FASTA - No Coverage Report
1367,SRR33943359,WGS,148.65,141682464,PRJNA1102327,SAMN49008954,Viral,52273363,USDA-NVSL,2025,...,USA,SRR33943359,2025-06-13_07-18-34,SRR33943359.fa,B3.13,"HA:ea1, PB1:am4, NA:ea1, MP:ea1, NP:am8, PA:ea...","ea1:22-003707-003:HA, am4:23-001855-001:PB1, e...","98.42%, 99.34%, 98.58%, 98.78%, 98.93%, 98.79%...","27, 15, 20, 12, 16, 26, 33, 8",Ran on FASTA - No Coverage Report
1368,SRR33943360,WGS,148.56,109583296,PRJNA1102327,SAMN49008953,Viral,40873907,USDA-NVSL,2025,...,USA,SRR33943360,2025-06-13_07-18-34,SRR33943360.fa,B3.13,"PB1:am4, MP:ea1, NP:am8, NA:ea1, PB2:am2.2, HA...","am4:23-001855-001:PB1, ea1:22-003707-003:MP, a...","99.12%, 98.98%, 98.93%, 98.72%, 98.51%, 98.30%...","20, 10, 16, 18, 34, 29, 22, 11",Ran on FASTA - No Coverage Report
1369,SRR33943301,WGS,148.82,146047742,PRJNA980729,SAMN49008919,Viral,51264801,USDA-NVSL,2025,...,USA,SRR33943301,2025-06-13_07-18-34,SRR33943301.fa,D1.1,"PB2:am24, NA:am4N1, MP:ea3, NS:ea3, PA:am4, HA...","am24:24-030039-001:PB2, am4N1:24-030039-001:NA...","99.56%, 99.04%, 100.00%, 98.93%, 98.75%, 99.47...","10, 10, 0, 9, 27, 9, 17, 7",Ran on FASTA - No Coverage Report


In [ ]:
# Get specific geolocation and name_state from genbank_mapping.tsv
os.chdir(metadata_folder)
genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")
genbank_mapping = genbank_mapping.rename(columns={"sra_run": "Run"})
# genbank_mapping["Run"] = genbank_mapping["sra_run"]
genbank_mapping.drop_duplicates(subset="Run", keep="first", inplace=True) # Drop duplicates
genbank_mapping["name_state"] = genbank_mapping["genbank_name"].apply(lambda x: x.split("/")[2]) # Get the name of the state

print(genbank_mapping)

metadata_genbank = pd.concat([metadata, genbank_mapping], join="inner")

print(metadata_genbank)

# print(metadata["name_state"])

# Get geolocation for second state attribute

os.chdir(home + "references/")
state_ref = pd.read_csv("states_ref.csv")
# Format: USA-[state abbreviation], e.g. USA-MD
metadata_genbank["Geo_Location"] = metadata_genbank["name_state"].apply(lambda x: 
                                                        # If "x" has the state abbreviation (e.g. "MD")
                                                        state_ref.loc[state_ref["Abbreviation"].str.contains('|'.join(x.replace(', ', ' ').split(' ')), regex=True), 'Country'].iloc[0] 
                                                        + "-" + 
                                                        x
                                                        if state_ref["Abbreviation"].str.contains("|".join((x.replace(", ", " ").split(" "))), regex=True).any()
                                                        # If "x" has the full state name (e.g. "Maryland")
                                                        else state_ref.loc[state_ref['State'].str.contains('|'.join(x.replace(', ', ' ').split(' ')), regex=True), 'Country'].iloc[0]
                                                        + "-" + 
                                                        state_ref.loc[state_ref['State'].str.contains('|'.join(x.replace(', ', ' ').split(' ')), regex=True), 'Abbreviation'].iloc[0] 
                                                        if state_ref["State"].str.contains("|".join((x.replace(", ", " ").split(" "))), regex=True).any() 
                                                        # If "x" has neither the state abbreviation nor the full state name
                                                        else 
                                                        x)

# Tests
print(state_ref.loc[state_ref['State'].str.contains('|'.join("Kentucky, whatever".replace(',', ' ').split(' ')), regex=True), 'Abbreviation'])
print(state_ref['State'].str.contains('|'.join("Kentucky, whatever".replace(', ', ' ').split(' ')), regex=True))
# print(metadata["name_state"])
print(metadata_genbank)
print(len(metadata))
metadata = metadata.merge(metadata_genbank, on="Run")
# metadata = pd.concat([metadata, metadata_genbank], join="inner")
# print(metadata[metadata["Geo_Location"] != "USA"])
display(metadata) 

                    seg_file  \
0      SRR28752446_HA_cns.fa   
8      SRR28752447_HA_cns.fa   
16     SRR28752448_HA_cns.fa   
24     SRR28752449_HA_cns.fa   
32     SRR28752450_HA_cns.fa   
...                      ...   
44878  SRR33764571_HA_cns.fa   
44886  SRR33764572_HA_cns.fa   
44894  SRR33764573_HA_cns.fa   
44902  SRR33764574_HA_cns.fa   
44910  SRR33764575_HA_cns.fa   

                                            seg_seq_name          Run seg  \
0      Consensus_SRR28752446_HA_cns_threshold_0.5_qua...  SRR28752446  HA   
8      Consensus_SRR28752447_HA_cns_threshold_0.5_qua...  SRR28752447  HA   
16     Consensus_SRR28752448_HA_cns_threshold_0.5_qua...  SRR28752448  HA   
24     Consensus_SRR28752449_HA_cns_threshold_0.5_qua...  SRR28752449  HA   
32     Consensus_SRR28752450_HA_cns_threshold_0.5_qua...  SRR28752450  HA   
...                                                  ...          ...  ..   
44878  Consensus_SRR33764571_HA_cns_threshold_0.5_qua...  SRR33764571  HA   

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,date,File Name,Genotype,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List,name_state_y,Geo_Location
0,SRR33125012,WGS,148.14,103837402,PRJNA1207547,SAMN47941494,Viral,38567600,USDA-NVSL,2025,...,2025-05-09_10-52-38,SRR33125012.fa,D1.1,"NP:am13, NS:ea3, PA:am4, PB2:am24, MP:ea3, PB1...","am13:24-030039-001:NP, ea3:22-013001-001:NS, a...","99.87%, 98.93%, 99.72%, 99.61%, 99.80%, 99.39%...","2, 9, 6, 9, 2, 11, 8, 12",Ran on FASTA - No Coverage Report,USA,USA
1,SRR33125013,WGS,148.47,151899011,PRJNA1207547,SAMN47941493,Viral,55900173,USDA-NVSL,2025,...,2025-05-09_10-52-46,SRR33125013.fa,D1.1,"NA:am4N1, NS:ea3, PB2:am24, MP:ea3, PB1:ea3, P...","am4N1:24-030039-001:NA, ea3:22-013001-001:NS, ...","98.87%, 99.17%, 99.83%, 99.90%, 99.16%, 99.58%...","13, 7, 4, 1, 19, 9, 9, 2",Ran on FASTA - No Coverage Report,USA,USA
2,SRR33125014,WGS,147.45,166656358,PRJNA1207547,SAMN47941492,Viral,60793010,USDA-NVSL,2025,...,2025-05-09_10-49-25,SRR33125014.fa,D1.1,"PB2:am24, MP:ea3, PB1:ea3, NP:am13, NA:am4N1, ...","am24:24-030039-001:PB2, ea3:22-013001-001:MP, ...","99.83%, 99.90%, 99.25%, 99.87%, 98.96%, 99.41%...","4, 1, 17, 2, 12, 10, 6, 11",Ran on FASTA - No Coverage Report,USA,USA
3,SRR33125015,WGS,148.33,107664413,PRJNA1207547,SAMN47941491,Viral,39666477,USDA-NVSL,2025,...,2025-05-09_10-48-50,SRR33125015.fa,D1.1,"PA:am4, NA:am4N1, NP:am13, PB1:ea3, NS:ea3, MP...","am4:24-030039-001:PA, am4N1:24-030039-001:NA, ...","99.58%, 99.03%, 99.87%, 98.99%, 99.28%, 99.90%...","9, 11, 2, 23, 6, 1, 3, 9",Ran on FASTA - No Coverage Report,USA,USA
4,SRR33125016,WGS,120.38,2476776,PRJNA1207547,SAMN47941490,Viral,986034,USDA-NVSL,2025,...,2025-05-09_10-52-46,SRR33125016.fa,D1.1,"NA:am4N1, PB1:ea3, NS:ea3, MP:ea3, NP:am13, PA...","am4N1:24-030039-001:NA, ea3:22-013001-001:PB1,...","99.62%, 99.28%, 99.28%, 99.69%, 99.93%, 98.34%...","4, 7, 6, 3, 1, 29, 9, 9",Ran on FASTA - No Coverage Report,USA,USA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1513,SRR33943358,WGS,148.58,145468815,PRJNA1102327,SAMN49008963,Viral,50309372,USDA-NVSL,2025,...,2025-06-13_07-18-34,SRR33943358.fa,B3.13,"PA:ea1, NA:ea1, PB2:am2.2, NP:am8, HA:ea1, PB1...","ea1:22-003707-003:PA, ea1:22-003707-003:NA, am...","98.84%, 98.65%, 98.51%, 98.86%, 98.12%, 99.56%...","25, 19, 34, 17, 32, 10, 9, 10",Ran on FASTA - No Coverage Report,USA,USA
1514,SRR33943359,WGS,148.65,141682464,PRJNA1102327,SAMN49008954,Viral,52273363,USDA-NVSL,2025,...,2025-06-13_07-18-34,SRR33943359.fa,B3.13,"HA:ea1, PB1:am4, NA:ea1, MP:ea1, NP:am8, PA:ea...","ea1:22-003707-003:HA, am4:23-001855-001:PB1, e...","98.42%, 99.34%, 98.58%, 98.78%, 98.93%, 98.79%...","27, 15, 20, 12, 16, 26, 33, 8",Ran on FASTA - No Coverage Report,USA,USA
1515,SRR33943360,WGS,148.56,109583296,PRJNA1102327,SAMN49008953,Viral,40873907,USDA-NVSL,2025,...,2025-06-13_07-18-34,SRR33943360.fa,B3.13,"PB1:am4, MP:ea1, NP:am8, NA:ea1, PB2:am2.2, HA...","am4:23-001855-001:PB1, ea1:22-003707-003:MP, a...","99.12%, 98.98%, 98.93%, 98.72%, 98.51%, 98.30%...","20, 10, 16, 18, 34, 29, 22, 11",Ran on FASTA - No Coverage Report,USA,USA
1516,SRR33943301,WGS,148.82,146047742,PRJNA980729,SAMN49008919,Viral,51264801,USDA-NVSL,2025,...,2025-06-13_07-18-34,SRR33943301.fa,D1.1,"PB2:am24, NA:am4N1, MP:ea3, NS:ea3, PA:am4, HA...","am24:24-030039-001:PB2, am4N1:24-030039-001:NA...","99.56%, 99.04%, 100.00%, 98.93%, 98.75%, 99.47...","10, 10, 0, 9, 27, 9, 17, 7",Ran on FASTA - No Coverage Report,USA,USA


In [6]:
# # If no states

# metadata_genbank = metadata

# metadata_genbank["name_state"] = "USA"

# metadata_genbank["Geo_Location"] = "USA"

# display(metadata_genbank)

## Collection Dates

If date is N/A, try finding it first. If saved dates are available, do NOT run the next cell. Comment it out and run the cell after. 

In [ ]:

# Get all dates
metadata["Collection_Date_Specific"] = metadata["BioSample"].apply(lambda x: search_collection_date(x, metadata) if "-" not in x else x) # Real dates have dashes

# Save this so we don't have to do it again

# os.chdir(temp_files)
metadata.to_csv("metadata_genbank.csv")

Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable t

In [10]:
# os.chdir(saved)

# metadata_to_merge = pd.read_csv("metadata_genbank_" + date_range + ".csv")
# metadata = pd.merge(metadata, metadata_to_merge, how="left")
# # metadata = pd.read_csv("metadata_genbank.csv")
# print(len(metadata))

If saved dates are available, un-comment and run the next cell

In [ ]:
# # # Upload saved data -- if doing this, make sure the above cell is commented out
# # os.chdir(temp_files + "saved/")
# # metadata_genbank = pd.read_csv("metadata_genbank.csv")
# # os.chdir(temp_files)

# # Get only updated dates

# # unknown_dates = metadata[(metadata["Collection_Date"] == "2024") | (metadata["Collection_Date"] == "2025")] # Dates we don't have

# def find_known_dates(x, df):
    
#     try:
#         date = metadata[metadata["BioSample"] == x]["Collection_Date"].values[0]
#     # print(date)
#         # print(date)
#         # if len(str(date)) == 4: # If this is just a year
#         #     date = search_collection_date(x, df)
#         # else:
#         date = dateutil.parser.parse(date, default=datetime(2000, 1, 1)) # .strftime("%Y-%m-%d") # If a date already exists
#         if date.day == dateutil.parser.parse("1/1/2000").day and date.month == dateutil.parser.parse("1/1/2000").month:
#             print("year only")
#             date = search_collection_date(x, df)
#         print("Success", date)
#     except:
#         date = search_collection_date(x, df) # If it's not parseable as a date
#     return date # If statement in lambda function will search for the "just year" values
        

# # years = ["2021", "2022", "2023", "2024", "2025"]
# # unknown_dates = metadata[metadata["Collection_Date"].isin(years)]
# # known_dates = metadata[~metadata["Collection_Date"].isin(years)]

# # Get new dates also 
# # new_dates = metadata["BioSample"].apply(lambda x: search_collection_date(x, metadata_genbank) if )

# updated_dates = metadata["BioSample"].apply(lambda x: find_known_dates(x, metadata)) # Update unknown dates, if possible
# metadata["Collection_Date"] = updated_dates

# # metadata = pd.concat([known_dates, unknown_dates], ignore_index=True, sort=True)

# # updated_unknown_dates = unknown_dates["BioSample"].apply(lambda x: search_collection_date(x, unknown_dates)) # Update unknown dates, if possible
# # unknown_dates["Collection_Date"] = updated_unknown_dates

# # metadata = pd.concat([known_dates, unknown_dates], ignore_index=True, sort=True)

# # Save this so we don't have to do it again

# # os.chdir(temp_files)
# # metadata.to_csv("metadata_genbank_" + date_range + ".csv")

# # display(metadata)

In [13]:
# os.chdir(temp_files)
# metadata.to_csv("metadata_genbank_" + date_range + ".csv")

# If no collection dates

# metadata_genbank["Collection_Date_Specific"] = metadata_genbank["Collection_Date"]

## Get host type

In [ ]:
# create a mask, where is True if the host does not exist
print(metadata["Host"])

mask = metadata["Host"].isna()

# choose between the original value and split isolate using the mask
metadata["Host"] = np.where(mask, metadata["isolate"].apply(lambda x: x if x != x or "/" not in x or len(x.split("/")) < 2 else x.split("/")[1]), metadata["Host"]) #  if "/" in metadata["isolate"] else metadata["Host"])
metadata["Host"] = metadata["Host"].apply(lambda x: x.lower() if x == x else x)

# print(metadata["Host"])

# Create animals ref if needed
unique_animals_all = sort_animals_andersen(metadata)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

# print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

os.chdir(home + "references/")

animals_ref = pd.read_csv("animals_ref.csv") # Upload animals ref

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# Unlikely for different_animals to be longer than the dataframe

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 

print(metadata["Host"])
# print(metadata["isolate"])

0       cattle
1       cattle
2       cattle
3       cattle
4       cattle
        ...   
135    chicken
136    chicken
137    chicken
138       duck
6             
Name: Host, Length: 125, dtype: object
[]
                      avian               cattle        feline   other_mammal  \
0          great_horned_owl            dairy_cow           cat     deer mouse   
1              common_raven               cattle  domestic_cat    house_mouse   
2             cooper's_hawk  cattle milk product     feral_cat          skunk   
3              coopers_hawk          bovine_milk        feline  striped_skunk   
4                   peafowl              bovine   domestic-cat     norway rat   
..                      ...                  ...           ...            ...   
797               raxorbill                  NaN           NaN            NaN   
798  von_schrenck's_bittern                  NaN           NaN            NaN   
799           harris's_hawk                  NaN           NaN  

In [36]:
# Get animals from animal reference
os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")
fix_animals_andersen(metadata, animals_ref) # Get host type

metadata["years"] = metadata["Collection_Date"].apply(lambda x: str(x).split("-")[0]) # Get year only from collection date

## Make names using all the attributes we collected

In [ ]:
for num, collection_date in enumerate(metadata["Collection_Date"]):
    if collection_date != collection_date: # if nan
         collection_date = "missing"
    try:
        print(collection_date)
        # If not a valid date, but has a year
        if collection_date.month == datetime.parser.parse("1/1/2000").month and collection_date.day == datetime.parser.parse("1/1/2000").day:
            year = collection_date.year
            metadata.loc[num, "Collection_Date"] = year
        else: # If actual date
            metadata.loc[num, "Collection_Date"] = collection_date
    except:
         metadata.loc[num, "Collection_Date"] = collection_date
        # else:
        #     try:
        #         parsed_date = dateutil.parser.parse(collection_date)
        #         date = parsed_date.strftime("%Y-%m-%d") # Make sure it doesn't default to today, if just a year
        #         metadata.loc[num, "Collection_Date"] = date
        #     except: # If no date at all
        #         metadata.loc[num, "Collection_Date"] = collection_date

    # metadata = metadata.dropna(thresh=2)



2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2024
2024
2024
2024
2025
2024
2024
2024
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2024
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2024
2024
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025
2024
2024
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025


In [ ]:
# Make names

metadata = metadata.fillna("") # Make sure the entire name does not become "NaN"

# + metadata["BioSample"] + "|" 
names = ">" + metadata["Run"] + "|" + "A/" + metadata["Host"] + "/" + metadata["name_state"] + "/" + metadata["Sample Name"] + "/" + metadata["years"].apply(lambda x: str(x)) + "|" + metadata["serotype"] + "|" + metadata["Geo_Location"] + "|" + metadata["Collection_Date"].apply(lambda x: str(x)) + "|" + metadata["Host_Type"] + "|" + metadata["Genotype"]

metadata["Name"] = names

# metadata_genbank.to_csv("metadata_genbank_named.csv")

display(metadata["Name"])

0      >SRR33993023|A/cattle/USA/25-016878-002/2025||...
1      >SRR33993024|A/cattle/USA/25-016866-004/2025||...
2      >SRR33993025|A/cattle/USA/25-016866-002/2025||...
3      >SRR33993026|A/cattle/USA/25-016847-007/2025||...
4      >SRR33993027|A/cattle/USA/25-016847-006/2025||...
                             ...                        
64                                      >|A////|||2025||
99                                      >|A////|||2024||
100                                     >|A////|||2025||
101                                     >|A////|||2025||
102                                     >|A////|||2025||
Name: Name, Length: 139, dtype: object

In [39]:
# Drop duplicate runs 
metadata = metadata.drop_duplicates(subset="Run", keep="first")

In [ ]:
print(metadata)
# metadata.to_csv("metadata_test.csv")

             Run Assay Type AvgSpotLen        Bases    BioProject  \
0    SRR33993023        WGS     148.15   68132847.0  PRJNA1102327   
1    SRR33993024        WGS     148.15   63105544.0  PRJNA1102327   
2    SRR33993025        WGS     148.28   46180111.0  PRJNA1102327   
3    SRR33993026        WGS     147.53   47042737.0  PRJNA1102327   
4    SRR33993027        WGS     147.76   65389924.0  PRJNA1102327   
..           ...        ...        ...          ...           ...   
135  SRR34270141        WGS     148.49  110079404.0   PRJNA980729   
136  SRR34270142        WGS     146.95  112507279.0   PRJNA980729   
137  SRR34270143        WGS     148.02  142063798.0   PRJNA980729   
138  SRR34270144        WGS     147.58  379458191.0   PRJNA980729   
6                                                                   

        BioSample BioSampleModel        Bytes Center Name Collection_Date  \
0    SAMN49104730          Viral   22640539.0   USDA-NVSL            2025   
1    SAMN49104729

## Make FASTA files

In [42]:
# Get information to create the fasta files

fasta_folder = originals + "avian-influenza/fasta/"

os.chdir(fasta_folder)

segments = ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]
pairs = []
fasta_files = {}

for genotype in genotypes: # ["B3.13", "D1.1"]:
    for segment in segments:
        pair = genotype + "_" + segment
        pairs.append(pair)

for pair in pairs:
    fasta_files[pair] = [] # List to hold fasta files

for run in metadata["Run"].values: # For each run 
    for dirpath, dirs, files in os.walk(fasta_folder): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            # print(file_name)
            if run in file_name: # Note that there will be ~8 files total with that run name
                # Make a fasta file and put it in the list
                with open(file_name) as f:
                    lines = f.readlines()
                    sequence = lines[1] 
                    # Each run/segment pair has one sequence -- it's placed into a file with other run/segment pairs with the same segment and genotype
                    header = metadata[metadata["Run"] == run].loc[:, "Name"].values[0]
                    genotype = metadata[metadata["Run"] == run].loc[:, "Genotype"].values[0]
                    # print(header)
                    # print(genotype)
                    # break 
                    segment = file_name.split("_")[-2]
                    # Find the pair that corresponds to 
                    pair_name = genotype + "_" + segment
                    this_specific_fasta = []
                    for pair in pairs:
                        # print(pair)
                        # print(pair_name)
                        if pair_name == pair:
                            this_specific_fasta.append(header)
                            this_specific_fasta.append(sequence)
                            fasta_files[pair].append(this_specific_fasta)
                f.close()
        break 

In [43]:
# print(fasta_files.keys())

In [ ]:
# Create fasta files 

os.chdir(originals + "complete/")
names = []
for pair in fasta_files.keys():
    output_path = originals + "complete/" + pair + "_" + date_range + "_andersen_updated.fasta"

    output_file = open(output_path, "w")
    for item in fasta_files[pair]:
        # for item in item:
        # item = fasta_files[pair]
        try:
            name = str(item[0].values[0]) # See if this is one we didn't have a collection date for
        except:
            name = str(item[0])
        print(name)
        names.append(name)
        # First is header, second is sequence
        # print(value)
        output_file.write(name + "\n")
        output_file.write(item[1])
    output_file.close()

print(len(names)/8)

>SRR34270263|A/red fox/USA/24-028269-001/2025||USA|2025|other_mammal|A3
>SRR34270263|A/red fox/USA/24-028269-001/2025||USA|2025|other_mammal|A3
>SRR34270263|A/red fox/USA/24-028269-001/2025||USA|2025|other_mammal|A3
>SRR34270263|A/red fox/USA/24-028269-001/2025||USA|2025|other_mammal|A3
>SRR34270263|A/red fox/USA/24-028269-001/2025||USA|2025|other_mammal|A3
>SRR34270263|A/red fox/USA/24-028269-001/2025||USA|2025|other_mammal|A3
>SRR34270263|A/red fox/USA/24-028269-001/2025||USA|2025|other_mammal|A3
>SRR34270263|A/red fox/USA/24-028269-001/2025||USA|2025|other_mammal|A3
>SRR34270253|A/red fox/USA/24-020141-001/2024||USA|2025|other_mammal|B3.2
>SRR34270253|A/red fox/USA/24-020141-001/2024||USA|2025|other_mammal|B3.2
>SRR34270253|A/red fox/USA/24-020141-001/2024||USA|2025|other_mammal|B3.2
>SRR34270253|A/red fox/USA/24-020141-001/2024||USA|2025|other_mammal|B3.2
>SRR34270253|A/red fox/USA/24-020141-001/2024||USA|2025|other_mammal|B3.2
>SRR34270253|A/red fox/USA/24-020141-001/2024||USA|202

## De-Duplication

In [ ]:
# De-duplication 

gisaid = downloads + "GISAID/complete/all_genotypes/" + date_range + "_all_genotypes_Antarctica_North_America_South_America/"

os.chdir(gisaid)

# Gisaid 
dfs_gisaid_list = []
dfs_gisaid = create_dataframes(gisaid)
dfs_gisaid_list.append(dfs_gisaid)

# dfs_gisaid = {}
# for df_gisaid in dfs_gisaid_list:
#     dfs_gisaid = dfs_gisaid | df_gisaid
# # dfs_gisaid2 = create_dataframes(gisaid2)

A3_HA
A3_MP
A3_NA
A3_NP
A3_NS
A3_PA
A3_PB1
A3_PB2
B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
B3.2_HA
B3.2_MP
B3.2_NA
B3.2_NP
B3.2_NS
B3.2_PA
B3.2_PB1
B3.2_PB2
B3.7_HA
B3.7_MP
B3.7_NA
B3.7_NP
B3.7_NS
B3.7_PA
B3.7_PB1
B3.7_PB2
C3.1_HA
C3.1_MP
C3.1_NA
C3.1_NP
C3.1_NS
C3.1_PA
C3.1_PB1
C3.1_PB2
D1.1_HA
D1.1_MP
D1.1_NA
D1.1_NP
D1.1_NS
D1.1_PA
D1.1_PB1
D1.1_PB2
Minor92_HA
Minor92_MP
Minor92_NA
Minor92_NP
Minor92_NS
Minor92_PA
Minor92_PB1
Minor92_PB2


In [ ]:
# Do the same with Andersen 

dfs_andersen = create_dataframes(originals + "complete/all_genotypes/" + date_range + "_all_genotypes/")

A1_HA
A1_MP
A1_NA
A1_NP
A1_NS
A1_PA
A1_PB1
A1_PB2
A2_HA
A2_MP
A2_NA
A2_NP
A2_NS
A2_PA
A2_PB1
A2_PB2
A3_HA
A3_MP
A3_NA
A3_NP
A3_NS
A3_PA
A3_PB1
A3_PB2
A4_HA
A4_MP
A4_NA
A4_NP
A4_NS
A4_PA
A4_PB1
A4_PB2
A5_HA
A5_MP
A5_NA
A5_NP
A5_NS
A5_PA
A5_PB1
A5_PB2
A6_HA
A6_MP
A6_NA
A6_NP
A6_NS
A6_PA
A6_PB1
A6_PB2
B1.1_HA
B1.1_MP
B1.1_NA
B1.1_NP
B1.1_NS
B1.1_PA
B1.1_PB1
B1.1_PB2
B1.2_HA
B1.2_MP
B1.2_NA
B1.2_NP
B1.2_NS
B1.2_PA
B1.2_PB1
B1.2_PB2
B1.3_HA
B1.3_MP
B1.3_NA
B1.3_NP
B1.3_NS
B1.3_PA
B1.3_PB1
B1.3_PB2
B2.1_HA
B2.1_MP
B2.1_NA
B2.1_NP
B2.1_NS
B2.1_PA
B2.1_PB1
B2.1_PB2
B2.2_HA
B2.2_MP
B2.2_NA
B2.2_NP
B2.2_NS
B2.2_PA
B2.2_PB1
B2.2_PB2
B3.10_HA
B3.10_MP
B3.10_NA
B3.10_NP
B3.10_NS
B3.10_PA
B3.10_PB1
B3.10_PB2
B3.11_HA
B3.11_MP
B3.11_NA
B3.11_NP
B3.11_NS
B3.11_PA
B3.11_PB1
B3.11_PB2
B3.12_HA
B3.12_MP
B3.12_NA
B3.12_NP
B3.12_NS
B3.12_PA
B3.12_PB1
B3.12_PB2
B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
B3.1_HA
B3.1_MP
B3.1_NA
B3.1_NP
B3.1_NS
B3.1_PA
B3.1_PB1
B3.1_PB2
B3.2_HA


In [50]:
for key in dfs_andersen.keys():
    dataframes = dfs_andersen[key]
    print(key)

print(dfs_andersen)

A1_HA
A1_MP
A1_NA
A1_NP
A1_NS
A1_PA
A1_PB1
A1_PB2
A2_HA
A2_MP
A2_NA
A2_NP
A2_NS
A2_PA
A2_PB1
A2_PB2
A3_HA
A3_MP
A3_NA
A3_NP
A3_NS
A3_PA
A3_PB1
A3_PB2
A4_HA
A4_MP
A4_NA
A4_NP
A4_NS
A4_PA
A4_PB1
A4_PB2
A5_HA
A5_MP
A5_NA
A5_NP
A5_NS
A5_PA
A5_PB1
A5_PB2
A6_HA
A6_MP
A6_NA
A6_NP
A6_NS
A6_PA
A6_PB1
A6_PB2
B1.1_HA
B1.1_MP
B1.1_NA
B1.1_NP
B1.1_NS
B1.1_PA
B1.1_PB1
B1.1_PB2
B1.2_HA
B1.2_MP
B1.2_NA
B1.2_NP
B1.2_NS
B1.2_PA
B1.2_PB1
B1.2_PB2
B1.3_HA
B1.3_MP
B1.3_NA
B1.3_NP
B1.3_NS
B1.3_PA
B1.3_PB1
B1.3_PB2
B2.1_HA
B2.1_MP
B2.1_NA
B2.1_NP
B2.1_NS
B2.1_PA
B2.1_PB1
B2.1_PB2
B2.2_HA
B2.2_MP
B2.2_NA
B2.2_NP
B2.2_NS
B2.2_PA
B2.2_PB1
B2.2_PB2
B3.10_HA
B3.10_MP
B3.10_NA
B3.10_NP
B3.10_NS
B3.10_PA
B3.10_PB1
B3.10_PB2
B3.11_HA
B3.11_MP
B3.11_NA
B3.11_NP
B3.11_NS
B3.11_PA
B3.11_PB1
B3.11_PB2
B3.12_HA
B3.12_MP
B3.12_NA
B3.12_NP
B3.12_NS
B3.12_PA
B3.12_PB1
B3.12_PB2
B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
B3.1_HA
B3.1_MP
B3.1_NA
B3.1_NP
B3.1_NS
B3.1_PA
B3.1_PB1
B3.1_PB2
B3.2_HA


In [51]:
print(len(list(dfs_gisaid.keys())))
print(len(list(dfs_andersen.keys())))

56
968


In [ ]:
# Merge dataframes and drop duplicates

full_dfs = defaultdict(list)

for i, andersen_key in enumerate(dfs_andersen.keys()):
    if len(dfs_andersen[andersen_key]) > 0:
        for j, gisaid_key in enumerate(dfs_gisaid.keys()):
            if andersen_key == gisaid_key:
                andersen_df = dfs_andersen[andersen_key][0]
                print(len(andersen_df))
                # print(andersen_df)
                gisaid_df = dfs_gisaid[gisaid_key][0]
                print(len(gisaid_df))
                # gisaid2_df = dfs_gisaid2[gisaid2_key][0]

                # print(pd.concat([gisaid_df, andersen_df]).drop_duplicates())

                full_df = pd.concat([andersen_df, gisaid_df], ignore_index=True)
                print("len full df:", len(full_df))
                # test = len(full_df.drop_duplicates(subset="isolate_partial"))

                dedup_df = full_df.drop_duplicates(subset="isolate_partial", keep="last")

                # print((full_df.loc[full_df.duplicated(subset="isolate_partial")]))
                # print((full_df.loc[full_df.duplicated(subset="isolate_partial")]))
                # print(full_df)
                
                # print("Keeping nothing: ", test)
                
                print("len deduplicated:", len(dedup_df))
                full_dfs[andersen_key].append(dedup_df)
            # else:
                # gisaid.add(gisaid_key)
                # andersen.add(andersen_key)
    
    # break 


# print(full_dfs)
# print(len(full_dfs))
# print(319*8)
# print(len(same))
# print(len(andersen))
# print(len(gisaid))

1
1
len full df: 2
len deduplicated: 1
1
1
len full df: 2
len deduplicated: 1
1
1
len full df: 2
len deduplicated: 1
1
1
len full df: 2
len deduplicated: 1
1
1
len full df: 2
len deduplicated: 1
1
1
len full df: 2
len deduplicated: 1
1
1
len full df: 2
len deduplicated: 1
1
1
len full df: 2
len deduplicated: 1
85
91
len full df: 176
len deduplicated: 108
85
91
len full df: 176
len deduplicated: 108
85
91
len full df: 176
len deduplicated: 108
85
91
len full df: 176
len deduplicated: 108
85
91
len full df: 176
len deduplicated: 108
85
91
len full df: 176
len deduplicated: 108
85
91
len full df: 176
len deduplicated: 108
85
91
len full df: 176
len deduplicated: 108
1
1
len full df: 2
len deduplicated: 1
1
1
len full df: 2
len deduplicated: 1
1
1
len full df: 2
len deduplicated: 1
1
1
len full df: 2
len deduplicated: 1
1
1
len full df: 2
len deduplicated: 1
1
1
len full df: 2
len deduplicated: 1
1
1
len full df: 2
len deduplicated: 1
1
1
len full df: 2
len deduplicated: 1
0
1
len full df:

In [53]:
# # If none in one database, only use the other and drop duplicates

# full_dfs = defaultdict(list)
# for key in dfs_andersen.keys():
#     print(key)
# # for key in ["D1.3"]:
#     dataframes = dfs_andersen[key]
#     for i, df in enumerate(dataframes):
#         print(i)
#         try:
#             full_df = df.merge(dfs_gisaid[key][i], how="outer")
#             # print(full_df)
#             full_df = full_df.drop_duplicates(subset=["isolate_partial"])
#             full_dfs[key].append(full_df)
#         except:
#             print("Failed to merge dataframes in ", key)
#             full_dfs[key].append(dataframes[i])

## Create FASTA files combining Andersen and GISAID

In [54]:
# Create FASTA files per segment

combined_files = downloads + "Combinations/GISAID_Andersen/" # B3_13_D1_1/" + date_range + "_B3_13_D1_1/"

os.chdir(combined_files)
for pair in full_dfs.keys():
    print(pair)
    output_path = combined_files + pair + "_combined_" + date_range + ".fasta" 

    output_file = open(output_path, "w")
    for item in full_dfs[pair]:
        # for item in item:
        # item = fasta_files[pair]
        for index, row in item.iterrows():
            name = item.loc[index, "full_header"]
            sequence = item.loc[index, "sequence"]
            # print(name)
        # First is header, second is sequence
        # print(value)
            output_file.write(name)
            output_file.write(sequence)
    output_file.close()

A3_HA
A3_MP
A3_NA
A3_NP
A3_NS
A3_PA
A3_PB1
A3_PB2
B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
B3.2_HA
B3.2_MP
B3.2_NA
B3.2_NP
B3.2_NS
B3.2_PA
B3.2_PB1
B3.2_PB2
B3.7_HA
B3.7_MP
B3.7_NA
B3.7_NP
B3.7_NS
B3.7_PA
B3.7_PB1
B3.7_PB2
C3.1_HA
C3.1_MP
C3.1_NA
C3.1_NP
C3.1_NS
C3.1_PA
C3.1_PB1
C3.1_PB2
D1.1_HA
D1.1_MP
D1.1_NA
D1.1_NP
D1.1_NS
D1.1_PA
D1.1_PB1
D1.1_PB2
Minor92_HA
Minor92_MP
Minor92_NA
Minor92_NP
Minor92_NS
Minor92_PA
Minor92_PB1
Minor92_PB2
